## 1. Verify the Python environment

checks which Python environment the notebook uses, prints
the TensorFlow version, and checks whether EfficientNetB0 is available.
It does not create or train a model.

In [1]:
import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("EfficientNetB0 available:", callable(tf.keras.applications.EfficientNetB0))

Python: c:\Users\SewwandiWijesinghe\SE4050-Deep-Learning-Assignment\venv\Scripts\python.exe
TensorFlow: 2.21.0
EfficientNetB0 available: True


## 2. Check dataset folders and class counts

locates the data folder and counts images in each class.
The dataset contains 5,600 Training images and 1,600 Testing images.
Each class has 1,400 Training images and 400 Testing images.

In [2]:
from pathlib import Path

# Find the data folder from either the project or notebooks folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data"
print("Dataset location:", data_dir)

classes = ["glioma", "meningioma", "notumor", "pituitary"]

for split in ["Training", "Testing"]:
    print(f"\n{split}:")
    for class_name in classes:
        folder = data_dir / split / class_name
        if not folder.is_dir():
            print(f"  {class_name}: FOLDER NOT FOUND")
        else:
            count = sum(
                1 for file in folder.iterdir()
                if file.is_file()
                and file.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
            print(f"  {class_name}: {count} images")

Dataset location: c:\Users\SewwandiWijesinghe\SE4050-Deep-Learning-Assignment\data

Training:
  glioma: 1400 images
  meningioma: 1400 images
  notumor: 1400 images
  pituitary: 1400 images

Testing:
  glioma: 400 images
  meningioma: 400 images
  notumor: 400 images
  pituitary: 400 images


## 3. Check image readability

This cell attempts to open and decode every image using Pillow.
Files that cannot be read are recorded for inspection.
All 7,200 images passed this check. No original files were modified.

In [3]:
from PIL import Image

image_files = sorted(
    file
    for split in ["Training", "Testing"]
    for file in (data_dir / split).rglob("*")
    if file.is_file()
    and file.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

unreadable_images = []

for file in image_files:
    try:
        with Image.open(file) as img:
            img.convert("RGB").load()
    except Exception as error:
        unreadable_images.append((str(file), str(error)))

print("Images checked:", len(image_files))
print("Unreadable images:", len(unreadable_images))

for file, error in unreadable_images[:10]:
    print(file, error)

Images checked: 7200
Unreadable images: 0


## 4. Check for exact duplicate files

In [4]:
import hashlib
from collections import defaultdict

files_by_hash = defaultdict(list)

for file in image_files:
    fingerprint = hashlib.sha256(file.read_bytes()).hexdigest()
    files_by_hash[fingerprint].append(file)

duplicate_groups = [
    files for files in files_by_hash.values()
    if len(files) > 1
]

train_test_overlap = [
    files for files in duplicate_groups
    if {file.relative_to(data_dir).parts[0] for file in files}
    == {"Training", "Testing"}
]

print("Groups of identical files:", len(duplicate_groups))
print("Groups appearing in both Training and Testing:",
      len(train_test_overlap))

for group in train_test_overlap[:5]:
    print("\nMatching files:")
    for file in group:
        print(" ", file.relative_to(data_dir))

Groups of identical files: 153
Groups appearing in both Training and Testing: 0


The check found 153 duplicate groups and no exact duplicates shared
between the Training and Testing folders.

This check does not detect every visually similar image or prove
that the folders contain scans from different patients.

## 5. Inspect duplicate locations and labels

This cell counts duplicate groups and extra copies separately within
Training and Testing. It also checks whether identical files have
different class labels.

These findings will guide consistent duplicate handling across the team.


In [5]:
# Count duplicate groups and extra copies in each folder.
for split in ["Training", "Testing"]:
    groups = [
        group for group in duplicate_groups
        if all(
            file.relative_to(data_dir).parts[0] == split
            for file in group
        )
    ]

    # Keep one copy per group when calculating extra copies.
    extra_copies = sum(len(group) - 1 for group in groups)

    print(f"\n{split}:")
    print("  Duplicate groups:", len(groups))
    print("  Extra copies:", extra_copies)

# Check whether identical files appear under different class labels.
conflicting_groups = [
    group for group in duplicate_groups
    if len({
        file.relative_to(data_dir).parts[1]
        for file in group
    }) > 1
]

print(
    "\nIdentical-image groups with different class labels:",
    len(conflicting_groups)
)


Training:
  Duplicate groups: 138
  Extra copies: 171

Testing:
  Duplicate groups: 15
  Extra copies: 16

Identical-image groups with different class labels: 0


## Dataset audit findings

- All 7,200 images were readable.
- Training contains 138 exact-duplicate groups and 171 extra copies.
- Testing contains 15 exact-duplicate groups and 16 extra copies.
- No exact file duplicates were found across Training and Testing.
- No exact-duplicate groups had conflicting class labels.
- Original files have not been modified.
- Shared duplicate handling and train/validation assignments must be
  settled before the final model comparison.
- These checks do not establish patient-level independence.

## 6. Check exact duplicates across training and validation

This check recreates the leader's split settings locally:
20% validation, seed 123, image size 224 × 224, and batch size 32.

It compares file hashes across the resulting training and validation
sets. A shared hash means an exact image copy appears in both sets.

This checks our local recreation. Confirming the leader's actual split
requires comparing their file lists or running the same check there.

In [6]:
# Recreate the split settings used in the leader's notebook.
split_settings = dict(
    directory=str(data_dir / "Training"),
    validation_split=0.2,
    seed=123,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
)

audit_train = tf.keras.utils.image_dataset_from_directory(
    subset="training",
    **split_settings
)

audit_val = tf.keras.utils.image_dataset_from_directory(
    subset="validation",
    **split_settings
)

# Match each file path to its previously calculated fingerprint.
hash_by_path = {
    file.resolve(): fingerprint
    for fingerprint, group in files_by_hash.items()
    for file in group
}

train_hashes = {
    hash_by_path[Path(file).resolve()]
    for file in audit_train.file_paths
}

val_hashes = {
    hash_by_path[Path(file).resolve()]
    for file in audit_val.file_paths
}

shared_hashes = train_hashes & val_hashes

print("\nTraining files:", len(audit_train.file_paths))
print("Validation files:", len(audit_val.file_paths))
print("Exact image contents shared across both sets:", len(shared_hashes))

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.

Training files: 4480
Validation files: 1120
Exact image contents shared across both sets: 56


In [7]:
# Show the filenames of one exact-duplicate group in Training.
example_group = next(
    group for group in duplicate_groups
    if all(
        file.relative_to(data_dir).parts[0] == "Training"
        for file in group
    )
)

for file in example_group:
    print(file.relative_to(data_dir))

# Directly compare the contents of the first two files.
print(
    "\nExactly the same file contents:",
    example_group[0].read_bytes() == example_group[1].read_bytes()
)

Training\meningioma\Tr-aug-me_89.jpg
Training\meningioma\Tr-me_889.jpg

Exactly the same file contents: True


## 8. Prepare a list of unique images

Keep one file path for each unique SHA-256 file hash.
This excludes extra exact copies from the proposed dataset list
without deleting or moving any original images.

The original Training and Testing groups remain separate.
This check identifies byte-identical files, not all visually similar images.

In [8]:
import pandas as pd

unique_rows = []

for fingerprint, group in files_by_hash.items():
    # Check that identical files belong to one source split and one class.
    source_splits = {
        file.relative_to(data_dir).parts[0] for file in group
    }
    labels = {
        file.relative_to(data_dir).parts[1] for file in group
    }

    assert len(source_splits) == 1, "Exact duplicate crosses Training/Testing."
    assert len(labels) == 1, "Exact duplicate has conflicting class labels."

    # Sort paths so the same file is selected each time.
    selected_file = sorted(
        group,
        key=lambda file: file.relative_to(data_dir).as_posix()
    )[0]

    relative_path = selected_file.relative_to(data_dir)

    unique_rows.append({
        "relative_path": relative_path.as_posix(),
        "source_split": relative_path.parts[0],
        "class_name": relative_path.parts[1],
        "sha256": fingerprint
    })

unique_df = pd.DataFrame(unique_rows).sort_values(
    "relative_path"
).reset_index(drop=True)

print("Unique files per original split:")
print(unique_df.groupby("source_split").size())
print("\nTotal unique files:", len(unique_df))
print("Extra exact copies excluded:", len(image_files) - len(unique_df))

Unique files per original split:
source_split
Testing     1584
Training    5429
dtype: int64

Total unique files: 7013
Extra exact copies excluded: 187


In [9]:
# Count unique images in each category.
print(
    pd.crosstab(
        unique_df["class_name"],
        unique_df["source_split"]
    ).reindex(columns=["Training", "Testing"])
)

source_split  Training  Testing
class_name                     
glioma            1400      386
meningioma        1386      398
notumor           1281      400
pituitary         1362      400


## 9. Create training, validation and testing lists

Split the unique images from the original Training folder into
80% training and 20% validation, using stratification to preserve
approximately the same class proportions.

Use random seed 123 for reproducibility.
Keep the unique images from the original Testing folder for final evaluation.

Verify that no exact file hash is shared between the three splits.
This does not establish patient independence or exclude near-duplicate images.

In [10]:
from sklearn.model_selection import train_test_split

# Separate the original Training and Testing lists.
training_pool = unique_df[
    unique_df["source_split"] == "Training"
].copy()

test_df = unique_df[
    unique_df["source_split"] == "Testing"
].copy()

# Split only the Training pool.
train_df, val_df = train_test_split(
    training_pool,
    test_size=0.20,
    random_state=123,
    stratify=training_pool["class_name"]
)

# Give each row its final split name.
train_df = train_df.copy()
val_df = val_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Combine the three lists into one shared table.
split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Check that all unique files are included exactly once.
assert len(split_df) == len(unique_df)
assert split_df["relative_path"].is_unique
assert split_df["sha256"].is_unique

print("Images per split:")
print(
    split_df["split"].value_counts().reindex(
        ["train", "validation", "test"]
    )
)

print("\nImages per class:")
print(
    pd.crosstab(
        split_df["class_name"],
        split_df["split"]
    ).reindex(columns=["train", "validation", "test"])
)

print("\nChecks passed: no exact duplicates across the splits.")

Images per split:
split
train         4343
validation    1086
test          1584
Name: count, dtype: int64

Images per class:
split       train  validation  test
class_name                         
glioma       1120         280   386
meningioma   1109         277   398
notumor      1025         256   400
pituitary    1089         273   400

Checks passed: no exact duplicates across the splits.


## 10. Save the shared dataset split

Save the selected image paths, class labels, SHA-256 hashes and
split assignments in a CSV file.

Paths are relative to the data folder, so teammates can use the
same list on their computers with the same dataset folder structure.

Original images are not moved or deleted.
All models should load this shared CSV for a consistent comparison.

In [11]:
# Use one consistent numeric label for each category.
class_to_id = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

split_df["class_id"] = split_df["class_name"].map(class_to_id)
assert split_df["class_id"].notna().all()

# Check that every selected image exists.
missing_files = [
    path for path in split_df["relative_path"]
    if not (data_dir / path).is_file()
]
assert not missing_files, f"Missing images: {missing_files[:5]}"

# Save outside data/ so the list can be committed to GitHub.
splits_dir = project_dir / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

manifest_path = splits_dir / "brain_tumor_split_seed123.csv"

columns = [
    "relative_path", "class_name", "class_id",
    "source_split", "split", "sha256"
]

manifest_df = split_df[columns].sort_values(
    ["split", "class_name", "relative_path"]
).reset_index(drop=True)

# Avoid silently replacing an existing, different split.
if manifest_path.exists():
    existing_df = pd.read_csv(manifest_path)
    pd.testing.assert_frame_equal(existing_df, manifest_df)
    print("Existing CSV already matches this split.")
else:
    manifest_df.to_csv(manifest_path, index=False)
    print("Shared split CSV saved.")

print("File:", manifest_path)
print("Rows:", len(manifest_df))
print("Class labels:", class_to_id)

Existing CSV already matches this split.
File: c:\Users\SewwandiWijesinghe\SE4050-Deep-Learning-Assignment\splits\brain_tumor_split_seed123.csv
Rows: 7013
Class labels: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
